# 🎮 TFT Raw Data Crawler
Crawl raw JSON data cho top 150 players và convert sang CSV (không thay đổi data)

Sử dụng thư viện `riotwatcher`

In [ ]:
# ========================================
# CELL 1: CONFIG + CHECKPOINT SYSTEM
# ========================================

import requests
import time
import json
import os
import pandas as pd
from datetime import datetime
from riotwatcher import TftWatcher, ApiError

# ⚠️ RIOT API KEY - Thay bằng API key của bạn
# Lấy tại: https://developer.riotgames.com/
API_KEY = "API-KEY-HERE"

# Khởi tạo TFT Watcher
watcher = TftWatcher(API_KEY)

PLATFORM = "vn2"
REGION_ACCOUNT = "asia"  # Asia region for account API
REGION_MATCH = "sea"  # SEA region for match data

# Lấy top 150 players có LP cao nhất
TOTAL_TOP_N = 150
MATCH_COUNT_PER_PLAYER = 200

# Output paths - Lưu trực tiếp trong thư mục chứa notebook
OUTPUT_DIR = "."

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RAW_JSON_PATH = f"{OUTPUT_DIR}/tft_raw_top{TOTAL_TOP_N}_{TIMESTAMP}.json"
CSV_LEADERBOARD_PATH = f"{OUTPUT_DIR}/tft_leaderboard_top{TOTAL_TOP_N}_{TIMESTAMP}.csv"
CSV_MATCHES_PATH = f"{OUTPUT_DIR}/tft_matches_top{TOTAL_TOP_N}_{TIMESTAMP}.csv"
CSV_PARTICIPANTS_PATH = f"{OUTPUT_DIR}/tft_participants_top{TOTAL_TOP_N}_{TIMESTAMP}.csv"

# ========================================
# 🔄 CHECKPOINT SYSTEM
# ========================================
CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint_tft_crawler.json"
CHECKPOINT_SAVE_INTERVAL = 1  # Save sau mỗi player

def save_checkpoint(raw_data, current_player_idx, phase="accounts"):
    """Lưu checkpoint để resume nếu bị disconnect"""
    checkpoint = {
        "timestamp": datetime.now().isoformat(),
        "phase": phase,  # "accounts" hoặc "matches"
        "current_player_idx": current_player_idx,
        "raw_data": raw_data
    }
    with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False)
    print(f"   💾 Checkpoint saved (player {current_player_idx}, phase: {phase})")

def load_checkpoint():
    """Load checkpoint nếu có"""
    if os.path.exists(CHECKPOINT_PATH):
        try:
            with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
            print(f"✅ Found checkpoint from {checkpoint['timestamp']}")
            print(f"   Phase: {checkpoint['phase']}, Player: {checkpoint['current_player_idx']}")
            return checkpoint
        except Exception as e:
            print(f"⚠️ Failed to load checkpoint: {e}")
    return None

def clear_checkpoint():
    """Xóa checkpoint sau khi hoàn thành"""
    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)
        print("🗑️ Checkpoint cleared!")

# Helper function cho Account API (TftWatcher không có account)
def get_account_by_puuid(puuid):
    url = f"https://{REGION_ACCOUNT}.api.riotgames.com/riot/account/v1/accounts/by-puuid/{puuid}"
    headers = {"X-Riot-Token": API_KEY}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    elif response.status_code == 429:
        retry_after = int(response.headers.get('Retry-After', 10))
        time.sleep(retry_after)
        return get_account_by_puuid(puuid)
    return None

print(f"✅ Config loaded!")
print(f"   API Key: {API_KEY[:20]}...")
print(f"   Platform: {PLATFORM}")
print(f"   Top N: {TOTAL_TOP_N}")
print(f"   Match count per player: {MATCH_COUNT_PER_PLAYER}")
print(f"   Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
# ========================================
# CELL 2: TEST API KEY
# ========================================

print("🔑 Testing API Key with riotwatcher...\n")

try:
    test_data = watcher.league.challenger(PLATFORM)
    print(f"✅ API Key hợp lệ!")
    print(f"   Challenger league có {len(test_data.get('entries', []))} entries")
except ApiError as e:
    print(f"❌ API Error: {e}")
    if e.response.status_code == 401:
        print("   → API Key không hợp lệ hoặc đã hết hạn!")
    elif e.response.status_code == 403:
        print("   → API Key không có quyền truy cập!")
    elif e.response.status_code == 429:
        print("   → Rate limited! Chờ một chút rồi thử lại.")

In [ ]:
# ========================================
# CELL 3: FETCH LEADERBOARD (RAW) + CHECK CHECKPOINT
# ========================================

print("🏆 Fetching TFT Leaderboard...\n")

# Kiểm tra checkpoint
checkpoint = load_checkpoint()

if checkpoint and checkpoint.get("raw_data"):
    print("🔄 Resuming from checkpoint...")
    raw_data = checkpoint["raw_data"]
    top_players = raw_data.get("players", [])
    print(f"   Loaded {len(raw_data.get('accounts', {}))} accounts")
    print(f"   Loaded {len(raw_data.get('matches', {}))} matches")
else:
    print("🆕 Starting fresh crawl...")
    raw_data = {
        "fetch_timestamp": datetime.now().isoformat(),
        "platform": PLATFORM,
        "leagues": {},
        "players": [],
        "accounts": {},
        "matches": {}
    }

    all_players = []

    try:
        # Fetch Challenger
        data = watcher.league.challenger(PLATFORM)
        raw_data["leagues"]["challenger"] = data
        entries = data.get('entries', [])
        for e in entries:
            e['tier'] = 'CHALLENGER'
            all_players.append(e)
        print(f"✅ Challenger: {len(entries)} players")
    except ApiError as e:
        print(f"❌ Challenger error: {e}")

    time.sleep(0.5)

    try:
        # Fetch Grandmaster
        data = watcher.league.grandmaster(PLATFORM)
        raw_data["leagues"]["grandmaster"] = data
        entries = data.get('entries', [])
        for e in entries:
            e['tier'] = 'GRANDMASTER'
            all_players.append(e)
        print(f"✅ Grandmaster: {len(entries)} players")
    except ApiError as e:
        print(f"❌ Grandmaster error: {e}")

    time.sleep(0.5)

    try:
        # Fetch Master
        data = watcher.league.master(PLATFORM)
        raw_data["leagues"]["master"] = data
        entries = data.get('entries', [])
        for e in entries:
            e['tier'] = 'MASTER'
            all_players.append(e)
        print(f"✅ Master: {len(entries)} players")
    except ApiError as e:
        print(f"❌ Master error: {e}")

    # Sort by LP and take top N
    all_players = sorted(all_players, key=lambda x: x.get('leaguePoints', 0), reverse=True)
    top_players = all_players[:TOTAL_TOP_N]
    raw_data["players"] = top_players

print(f"\n🏆 Selected top {len(top_players)} players!")

In [ ]:
# ========================================
# CELL 4: FETCH ACCOUNT DATA (RAW) + CHECKPOINT
# ========================================

print("📋 Fetching Account Data...\n")

# Xác định điểm bắt đầu từ checkpoint
start_idx = 0
if checkpoint and checkpoint.get("phase") in ["accounts", "matches"]:
    if checkpoint.get("phase") == "accounts":
        start_idx = checkpoint.get("current_player_idx", 0)
        print(f"🔄 Resuming accounts from player {start_idx + 1}")
    else:
        # Đã qua phase accounts, skip toàn bộ
        start_idx = len(top_players)
        print(f"✅ Accounts already completed, skipping...")

for i, player in enumerate(top_players, 1):
    # Skip những player đã fetch rồi
    if i - 1 < start_idx:
        continue
        
    puuid = player.get('puuid')
    print(f"⏳ [{i}/{len(top_players)}] ", end="")
    
    if puuid:
        # Kiểm tra đã có account chưa
        if puuid in raw_data.get("accounts", {}):
            account = raw_data["accounts"][puuid]
            print(f"⏭️ Already have {account.get('gameName', 'Unknown')}#{account.get('tagLine', 'VN2')}")
            continue
            
        account_data = get_account_by_puuid(puuid)
        if account_data:
            raw_data["accounts"][puuid] = account_data
            game_name = account_data.get('gameName', 'Unknown')
            tag_line = account_data.get('tagLine', 'VN2')
            print(f"✅ {game_name}#{tag_line}")
        else:
            print(f"❌ Failed to fetch account")
    else:
        print(f"❌ No PUUID")
    
    # Save checkpoint sau mỗi CHECKPOINT_SAVE_INTERVAL players
    if i % CHECKPOINT_SAVE_INTERVAL == 0:
        save_checkpoint(raw_data, i, phase="accounts")
    
    time.sleep(0.3)

# Save checkpoint cuối phase accounts
save_checkpoint(raw_data, len(top_players), phase="accounts")
print(f"\n✅ Got {len(raw_data['accounts'])} accounts!")

In [ ]:
# ========================================
# CELL 5: FETCH MATCH HISTORY (RAW) + CHECKPOINT
# ========================================

print(f"📊 Fetching Match History ({MATCH_COUNT_PER_PLAYER} matches/player)...\n")

# Xác định điểm bắt đầu từ checkpoint
start_idx = 0
if checkpoint and checkpoint.get("phase") == "matches":
    start_idx = checkpoint.get("current_player_idx", 0)
    print(f"🔄 Resuming matches from player {start_idx + 1}")
    print(f"   Already have {len(raw_data.get('matches', {}))} matches\n")

total_matches = len(raw_data.get("matches", {}))

for i, player in enumerate(top_players, 1):
    # Skip những player đã fetch rồi
    if i - 1 < start_idx:
        continue
        
    puuid = player.get('puuid')
    if not puuid:
        continue
    
    account = raw_data["accounts"].get(puuid, {})
    riot_id = f"{account.get('gameName', 'Unknown')}#{account.get('tagLine', 'VN2')}"
    
    print(f"🎮 [{i}/{len(top_players)}] {riot_id} ", end="")
    
    try:
        # Get match IDs
        match_ids = watcher.match.by_puuid(REGION_MATCH, puuid, count=MATCH_COUNT_PER_PLAYER)
        
        if not match_ids:
            print("⚠️ No matches")
            # Save checkpoint
            save_checkpoint(raw_data, i, phase="matches")
            continue
        
        matches_fetched = 0
        matches_skipped = 0
        
        for match_id in match_ids:
            # Skip match đã có rồi (từ checkpoint hoặc player khác)
            if match_id in raw_data["matches"]:
                matches_skipped += 1
                continue
                
            try:
                match_detail = watcher.match.by_id(REGION_MATCH, match_id)
                raw_data["matches"][match_id] = match_detail
                matches_fetched += 1
            except ApiError as e:
                if e.response.status_code == 429:
                    retry_after = int(e.response.headers.get('Retry-After', 10))
                    print(f"\n   ⏳ Rate limited, waiting {retry_after}s...", end="")
                    time.sleep(retry_after)
                    try:
                        match_detail = watcher.match.by_id(REGION_MATCH, match_id)
                        raw_data["matches"][match_id] = match_detail
                        matches_fetched += 1
                    except:
                        pass
            
            time.sleep(0.25)
        
        total_matches += matches_fetched
        print(f"✅ {matches_fetched} new, {matches_skipped} skipped")
        
    except ApiError as e:
        if e.response.status_code == 429:
            retry_after = int(e.response.headers.get('Retry-After', 10))
            print(f"⏳ Rate limited, waiting {retry_after}s...")
            time.sleep(retry_after)
        else:
            print(f"❌ Error: {e}")
    
    # Save checkpoint sau mỗi player
    save_checkpoint(raw_data, i, phase="matches")

print(f"\n✅ Total unique matches: {len(raw_data['matches'])}")

In [ ]:
# ========================================
# CELL 6: SAVE RAW JSON + CLEAR CHECKPOINT
# ========================================

print("💾 Saving Raw JSON...\n")

with open(RAW_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(raw_data, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(RAW_JSON_PATH) / 1024 / 1024
print(f"✅ Saved: {RAW_JSON_PATH}")
print(f"   Size: {file_size:.2f} MB")

# Xóa checkpoint sau khi hoàn thành
clear_checkpoint()

In [ ]:
# ========================================
# CELL 7: CONVERT JSON TO CSV
# ========================================

print("📊 Converting JSON to CSV...\n")

# Helper: Build puuid -> riot_id mapping
def get_riot_id(puuid):
    account = raw_data.get("accounts", {}).get(puuid, {})
    if account:
        return f"{account.get('gameName', 'Unknown')}#{account.get('tagLine', 'VN2')}"
    return None

# 1. LEADERBOARD CSV
leaderboard_rows = []
for i, player in enumerate(raw_data.get("players", []), 1):
    puuid = player.get('puuid')
    account = raw_data.get("accounts", {}).get(puuid, {})
    
    row = {
        "rank": i,
        "riot_id": get_riot_id(puuid),
        "summonerId": player.get("summonerId"),
        "puuid": puuid,
        "leaguePoints": player.get("leaguePoints"),
        "wins": player.get("wins"),
        "losses": player.get("losses"),
        "tier": player.get("tier"),
        "veteran": player.get("veteran"),
        "inactive": player.get("inactive"),
        "freshBlood": player.get("freshBlood"),
        "hotStreak": player.get("hotStreak"),
        "gameName": account.get("gameName"),
        "tagLine": account.get("tagLine"),
    }
    leaderboard_rows.append(row)

leaderboard_df = pd.DataFrame(leaderboard_rows)
leaderboard_df.to_csv(CSV_LEADERBOARD_PATH, index=False, encoding='utf-8-sig')
print(f"✅ Leaderboard: {CSV_LEADERBOARD_PATH} ({len(leaderboard_df)} rows)")

# 2. MATCHES CSV
matches_rows = []
for match_id, match_data in raw_data.get("matches", {}).items():
    metadata = match_data.get("metadata", {})
    info = match_data.get("info", {})
    
    row = {
        "match_id": match_id,
        "data_version": metadata.get("data_version"),
        "participant_puuids": json.dumps(metadata.get("participants", [])),
        "game_datetime": info.get("game_datetime"),
        "game_length": info.get("game_length"),
        "game_version": info.get("game_version"),
        "queue_id": info.get("queue_id"),
        "tft_game_type": info.get("tft_game_type"),
        "tft_set_core_name": info.get("tft_set_core_name"),
        "tft_set_number": info.get("tft_set_number"),
    }
    matches_rows.append(row)

matches_df = pd.DataFrame(matches_rows)
matches_df.to_csv(CSV_MATCHES_PATH, index=False, encoding='utf-8-sig')
print(f"✅ Matches: {CSV_MATCHES_PATH} ({len(matches_df)} rows)")

# 3. PARTICIPANTS CSV
participants_rows = []
for match_id, match_data in raw_data.get("matches", {}).items():
    info = match_data.get("info", {})
    
    for participant in info.get("participants", []):
        puuid = participant.get("puuid")
        row = {
            "match_id": match_id,
            "puuid": puuid,
            "riot_id": get_riot_id(puuid),
            "placement": participant.get("placement"),
            "level": participant.get("level"),
            "gold_left": participant.get("gold_left"),
            "last_round": participant.get("last_round"),
            "players_eliminated": participant.get("players_eliminated"),
            "time_eliminated": participant.get("time_eliminated"),
            "total_damage_to_players": participant.get("total_damage_to_players"),
            "companion": json.dumps(participant.get("companion")),
            "augments": json.dumps(participant.get("augments", [])),
            "traits": json.dumps(participant.get("traits", [])),
            "units": json.dumps(participant.get("units", [])),
        }
        participants_rows.append(row)

participants_df = pd.DataFrame(participants_rows)
participants_df.to_csv(CSV_PARTICIPANTS_PATH, index=False, encoding='utf-8-sig')
print(f"✅ Participants: {CSV_PARTICIPANTS_PATH} ({len(participants_df)} rows)")

print("\n🎉 Done!")

In [ ]:
# ========================================
# CELL 8: PREVIEW DATA
# ========================================

print("📊 Data Preview\n")

print("🏆 Leaderboard:")
display(leaderboard_df[['rank', 'riot_id', 'tier', 'leaguePoints', 'wins', 'losses']].head(10))

print("\n📊 Matches:")
display(matches_df.head(5))

print("\n👥 Participants:")
display(participants_df[['match_id', 'riot_id', 'placement', 'level', 'augments']].head(10))